In [1]:
# Install duckdb and pandas
!pip install duckdb pandas --quiet

# Import required libraries
import duckdb
import pandas as pd
import os
import random
import numpy as np
import datetime

# Create an in-memory DuckDB connection
conn = duckdb.connect(database=':memory:')

# Create a test table with sample data
conn.execute("""
    CREATE TABLE hello_world (
        id INTEGER,
        message VARCHAR,
        created_at TIMESTAMP
    )
""")
conn.execute("""
    INSERT INTO hello_world VALUES
    (1, 'Hello, World!', NOW()),
    (2, 'This is Kata 7.1', NOW()),
    (3, 'Great success!', NOW())
""")

# Run a SELECT query and print the results
df = conn.execute("SELECT * FROM hello_world").df()
print(df)

# Confirm environment is ready
print("Environment ready ✓")

   id           message                 created_at
0   1     Hello, World! 2026-11-09 15:11:52.686563
1   2  This is Kata 7.1 2026-11-09 15:11:52.686563
2   3    Great success! 2026-11-09 15:11:52.686563
Environment ready ✓


In [1]:
import numpy as np
import pandas as pd
import random
import os
from datetime import datetime, timedelta

# Встановлюємо seed для повторення результатів
random.seed(42)
np.random.seed(42)

# Створюємо директорію, якщо її не існує
os.makedirs("bronze", exist_ok=True)

# Функція для генерації дат у змішаних форматах
def generate_random_date():
    start_date = datetime(2024, 1, 1)
    random_days = random.randint(0, 364)
    date = start_date + timedelta(days=random_days)
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%b %d %Y"]
    return date.strftime(random.choice(formats))

# Формуємо 500 рядків даних
num_rows = 500
data = []

for _ in range(num_rows):
    order_id = f"ORD-{random.randint(10000, 99999)}"
    if random.random() < 0.03:  # 3% дублікати
        order_id = random.choice(data)[0] if data else order_id

    customer_id = random.randint(1000, 9999)
    region = random.choice(["North", "South", "East", "West"])
    order_date = generate_random_date()
    product_category = random.choice(["Electronics", "Clothing", "Food", "Home", "Sports"])
    
    # 5% null значень, 2% від'ємні значення
    if random.random() < 0.05:
        amount = None
    elif random.random() < 0.02:
        amount = round(-abs(random.uniform(10, 500)), 2)
    else:
        amount = round(random.uniform(10, 500), 2)
    
    quantity = random.randint(1, 10)
    status = random.choices(["completed", "returned", "pending"], weights=[80, 15, 5])[0]

    data.append([
        order_id, customer_id, region, order_date, product_category, amount, quantity, status
    ])

# Створюємо датафрейм і записуємо у CSV
df = pd.DataFrame(data, columns=[
    "order_id", "customer_id", "region", "order_date",
    "product_category", "amount", "quantity", "status"
])

# Зберігаємо у CSV
file_path = "bronze/transactions_raw.csv"
df.to_csv(file_path, index=False)

# Виводимо статистику про якість даних
null_count = df['amount'].isnull().sum()
duplicate_count = df['order_id'].duplicated().sum()
unique_date_formats = len(set(df['order_date'].apply(
    lambda x: datetime.strptime(x, "%Y-%m-%d").strftime("%Y-%m-%d") 
    if "-" in x else x.split("/")[-1] if "/" in x else x
)))

print(f"✅ Dataset збережено в {file_path}")
print(f"Загальна кількість рядків: {len(df)}")
print(f"Кількість порожніх значень у 'amount': {null_count}")
print(f"Кількість дублікатів 'order_id': {duplicate_count}")
print(f"Кількість унікальних форматів дати: {unique_date_formats}")

✅ Dataset збережено в bronze/transactions_raw.csv
Загальна кількість рядків: 500
Кількість порожніх значень у 'amount': 22
Кількість дублікатів 'order_id': 16
Кількість унікальних форматів дати: 279


In [2]:
import duckdb

# Під'єднуємося до DuckDB
conn = duckdb.connect(database=':memory:')

# Завантажуємо CSV у DuckDB
conn.execute("CREATE TABLE transactions AS SELECT * FROM read_csv_auto('bronze/transactions_raw.csv')")

# SQL-запит для профілювання
query = """
    SELECT 
        COUNT(*) AS row_count,
        COUNT(CASE WHEN amount IS NULL THEN 1 ELSE NULL END) AS null_in_amount,
        COUNT(DISTINCT order_id) AS unique_order_ids,
        COUNT(order_id) - COUNT(DISTINCT order_id) AS duplicate_order_ids,
        MIN(amount) AS min_amount,
        MAX(amount) AS max_amount,
        COUNT(DISTINCT status) AS unique_status_values,
        COUNT(DISTINCT order_date) AS unique_date_formats
    FROM transactions
"""
profile = conn.execute(query).fetchdf()
print(profile)

   row_count  null_in_amount  unique_order_ids  duplicate_order_ids  \
0        500              22               484                   16   

   min_amount  max_amount  unique_status_values  unique_date_formats  
0     -438.64      498.92                     3                  405  
